# SFT Training — Quartermaster Environment

Fine-tune **Qwen3-4B-Instruct** on inventory management trajectories using Unsloth (efficient LoRA) + TRL SFTTrainer.

**Input:** `sft_data.jsonl` (generated by `generate_sft_data.ipynb`)

**Output:**
- `./sft_model/` — LoRA adapter + tokenizer
- `./sft_model_merged/` — merged 16-bit weights for GRPO continuation

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install unsloth trl datasets matplotlib python-dotenv

## 1. Configuration

In [ ]:
import os
import json
import time
import logging

from dotenv import load_dotenv
load_dotenv()

# --- Config ---
SFT_DATA_FILE = os.getenv("SFT_DATA_FILE", "sft_data.jsonl")
MIN_REWARD = float(os.getenv("MIN_REWARD", "-999"))  # Set higher to filter bad examples
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "./sft_model")
NUM_EPOCHS = int(os.getenv("NUM_EPOCHS", "3"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "2"))
GRAD_ACCUM = int(os.getenv("GRAD_ACCUM", "4"))
MAX_SEQ_LENGTH = int(os.getenv("MAX_SEQ_LENGTH", "4096"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE", "1e-4"))
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("sft_train")

print(f"Model: {MODEL_NAME}")
print(f"Data: {SFT_DATA_FILE}")
print(f"Min reward filter: {MIN_REWARD}")
print(f"Epochs: {NUM_EPOCHS}, batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}")
print(f"Max seq length: {MAX_SEQ_LENGTH}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Output: {OUTPUT_DIR}")

## 2. Load & Filter Dataset

Loads `sft_data.jsonl`, optionally filters by minimum reward, and converts to HuggingFace Dataset with `messages` format.

In [ ]:
from datasets import Dataset

examples = []
total_loaded = 0
total_filtered = 0
reward_values = []

with open(SFT_DATA_FILE) as f:
    for line in f:
        ex = json.loads(line.strip())
        total_loaded += 1
        reward_values.append(ex["reward"])
        if ex["reward"] >= MIN_REWARD:
            examples.append({
                "messages": [
                    {"role": "system", "content": ex["system"]},
                    {"role": "user", "content": ex["user"]},
                    {"role": "assistant", "content": ex["assistant"]},
                ]
            })
        else:
            total_filtered += 1

avg_reward = sum(reward_values) / len(reward_values) if reward_values else 0
dataset = Dataset.from_list(examples)

print(f"Loaded {total_loaded} total, kept {len(examples)} (filtered {total_filtered})")
print(f"Reward stats: avg={avg_reward:.3f}, min={min(reward_values):.3f}, max={max(reward_values):.3f}")
print(f"Dataset size: {len(dataset)}")

## 3. Load Model with Unsloth

Load Qwen3-4B-Instruct in 4-bit quantization for memory-efficient training.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

print(f"Model loaded: {MODEL_NAME}")

## 4. Apply LoRA Adapters

Apply LoRA to attention + MLP layers with Unsloth's optimized gradient checkpointing.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 5. Configure SFT Trainer

Using TRL's SFTTrainer with `assistant_only_loss=True` — trains only on assistant responses, not system/user prompts.
Chat template is auto-applied from the tokenizer when dataset has a `messages` column.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    max_seq_length=MAX_SEQ_LENGTH,
    seed=42,
    report_to="none",
    assistant_only_loss=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

print("SFT Trainer configured. Ready to train.")

## 6. Train

In [ ]:
train_start = time.time()
log.info("Starting SFT training...")

trainer.train()

train_time = time.time() - train_start
log.info(f"SFT training finished in {train_time:.1f}s ({train_time/60:.1f}min)")

## 7. Save Model

Save both the LoRA adapter and a merged 16-bit model (needed for GRPO continuation).

In [ ]:
# Save LoRA adapter + tokenizer
log.info(f"Saving LoRA adapters to {OUTPUT_DIR}")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save merged 16-bit model for GRPO continuation
merged_dir = OUTPUT_DIR.rstrip("/") + "_merged"
log.info(f"Saving merged model to {merged_dir}")
model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")

print(f"LoRA saved to: {OUTPUT_DIR}")
print(f"Merged model saved to: {merged_dir}")

## 8. Training Curves

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def _smooth(values, window):
    smoothed = []
    for i in range(len(values)):
        start = max(0, i - window)
        smoothed.append(sum(values[start:i+1]) / (i - start + 1))
    return smoothed

history = trainer.state.log_history
steps = [h["step"] for h in history if "loss" in h]
losses = [h["loss"] for h in history if "loss" in h]
lr_steps = [h["step"] for h in history if "learning_rate" in h]
lrs = [h["learning_rate"] for h in history if "learning_rate" in h]

if steps:
    window = max(5, len(losses) // 20) if len(losses) > 10 else 1

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("SFT Training — QuarterMaster", fontsize=16, fontweight="bold")

    # Loss curve
    ax = axes[0, 0]
    ax.plot(steps, losses, color="#3b82f6", linewidth=1, alpha=0.4, label="Raw loss")
    if len(losses) > 10:
        ax.plot(steps, _smooth(losses, window), color="#ef4444", linewidth=2.5, label=f"Smoothed (w={window})")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Learning rate
    ax = axes[0, 1]
    if lr_steps:
        ax.plot(lr_steps, lrs, color="#a855f7", linewidth=2)
        ax.ticklabel_format(style="scientific", axis="y", scilimits=(0, 0))
    ax.set_xlabel("Step")
    ax.set_ylabel("Learning Rate")
    ax.set_title("Learning Rate Schedule")
    ax.grid(True, alpha=0.3)

    # Loss histogram
    ax = axes[1, 0]
    ax.hist(losses, bins=40, color="#06b6d4", alpha=0.7, edgecolor="#000")
    ax.axvline(x=sum(losses)/len(losses), color="#ef4444", linestyle="--", linewidth=2,
               label=f"Mean={sum(losses)/len(losses):.4f}")
    ax.set_xlabel("Loss")
    ax.set_ylabel("Count")
    ax.set_title("Loss Distribution")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Loss by epoch
    ax = axes[1, 1]
    epoch_data = [h for h in history if "epoch" in h and "loss" in h]
    if epoch_data:
        epoch_groups = {}
        for h in epoch_data:
            ep_int = int(h["epoch"])
            epoch_groups.setdefault(ep_int, []).append(h["loss"])
        ep_labels = sorted(epoch_groups.keys())
        ep_means = [sum(epoch_groups[e])/len(epoch_groups[e]) for e in ep_labels]
        colors = ["#3b82f6", "#22c55e", "#f59e0b", "#ef4444", "#a855f7"][:len(ep_labels)]
        ax.bar([f"Epoch {e+1}" for e in ep_labels], ep_means, color=colors, width=0.5)
        ax.set_ylabel("Mean Loss")
    ax.set_title("Mean Loss per Epoch")
    ax.grid(True, alpha=0.3, axis="y")

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    fig.savefig(os.path.join(OUTPUT_DIR, "sft_training_dashboard.png"), dpi=150)
    plt.show()
    print(f"Dashboard saved to {OUTPUT_DIR}/sft_training_dashboard.png")
else:
    print("No training loss data found.")

## 9. Training Data Reward Analysis

In [ ]:
reward_by_task = {}
try:
    with open(SFT_DATA_FILE) as f:
        for line in f:
            ex = json.loads(line.strip())
            if ex.get("reward") is not None and ex["reward"] >= MIN_REWARD:
                task = ex.get("task", "unknown")
                reward_by_task.setdefault(task, {"rewards": [], "days": []})
                reward_by_task[task]["rewards"].append(ex["reward"])
                reward_by_task[task]["days"].append(ex.get("day", 0))
except Exception as e:
    print(f"Could not load data: {e}")

if reward_by_task:
    task_colors = {"easy": "#22c55e", "medium": "#f59e0b", "hard": "#ef4444"}
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("SFT Training Data — Reward Analysis", fontsize=14, fontweight="bold")

    # Reward by day
    ax = axes[0]
    for task in ["easy", "medium", "hard"]:
        if task in reward_by_task:
            paired = sorted(zip(reward_by_task[task]["days"], reward_by_task[task]["rewards"]))
            ax.plot([p[0] for p in paired], [p[1] for p in paired],
                    color=task_colors.get(task, "#888"), alpha=0.6, linewidth=1, label=task)
    ax.set_xlabel("Day")
    ax.set_ylabel("Reward")
    ax.set_title("Reward by Day")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Reward histogram
    ax = axes[1]
    for task in ["easy", "medium", "hard"]:
        if task in reward_by_task:
            ax.hist(reward_by_task[task]["rewards"], bins=30, alpha=0.5,
                    color=task_colors.get(task, "#888"), label=task, edgecolor="#000")
    ax.set_xlabel("Reward")
    ax.set_ylabel("Count")
    ax.set_title("Reward Distribution")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Mean reward bar
    ax = axes[2]
    labels, means, colors = [], [], []
    for task in ["easy", "medium", "hard"]:
        if task in reward_by_task:
            r = reward_by_task[task]["rewards"]
            labels.append(f"{task}\n(n={len(r)})")
            means.append(sum(r) / len(r))
            colors.append(task_colors.get(task, "#888"))
    ax.bar(labels, means, color=colors, width=0.5)
    ax.set_ylabel("Mean Reward")
    ax.set_title("Mean Reward by Task")
    ax.grid(True, alpha=0.3, axis="y")

    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(os.path.join(OUTPUT_DIR, "sft_data_rewards.png"), dpi=150)
    plt.show()
    print(f"Reward analysis saved to {OUTPUT_DIR}/sft_data_rewards.png")
else:
    print("No reward data available.")

## 10. Save Training Log

In [ ]:
curve_data = {
    "steps": steps, "losses": losses,
    "lr_steps": lr_steps, "lrs": lrs,
}
data_path = os.path.join(OUTPUT_DIR, "sft_training_log.json")
with open(data_path, "w") as f:
    json.dump(curve_data, f, indent=2)
print(f"Training log saved to {data_path}")
print(f"\nSFT training complete. Next step: run grpo_training.ipynb")